# Cellpose finetuning
A notebook to evaluate finetuning performance of various cellpose fine-tuned models

In [ ]:
import os
from pathlib import Path
import re
from sphero_vem.io import imread_labels_downscaled
from dotenv import load_dotenv
import cellpose.metrics as metrics
import numpy as np
import tifffile
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

load_dotenv("../.env")
DATA_ROOT = Path(os.environ.get("DATA_ROOT"))
dir_gt = DATA_ROOT / "processed/labeled/Au_01-vol_01/labeled-01"
dir_pred = DATA_ROOT / "processed/segmented/finetuning"
dir_pretrained = DATA_ROOT / "processed/segmented/Au_01-vol_01/downscaling_tests"
dir_save = DATA_ROOT / "figures/segmentation"
dir_save.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.family"] = "Helvetica"

In [ ]:
# Functions
def calculate_ap(row: pd.Series) -> np.ndarray:
    ground_truth = imread_labels_downscaled(
        row["ground_truth_path"], row["downscaling_factor"]
    )
    predictions = tifffile.imread(row["prediction_path"])
    metric = metrics.average_precision(
        ground_truth, predictions, threshold=row["iou_threshold"]
    )[0]
    return metric

In [ ]:
# Tuple of (model_name, downscaling, ground truth, masks)
targets = []

# Log also ground truths to track the unique ones used before building up df
gt_paths = []

for dir_model in dir_pred.glob("*/"):
    factor_train = re.search(r"ds(\d+)", dir_model.name).groups()[0]
    for masks_path in dir_model.rglob("*.tif"):
        ds_match = re.search(r"downscaled-(\d+)", masks_path.parent.name)
        factor = ds_match.groups()[0] if ds_match else factor_train
        gt_paths.append(dir_gt / "labels" / masks_path.name)
        targets.append(
            (
                dir_model.name,
                int(factor_train),
                int(factor),
                gt_paths[-1],
                masks_path,
            )
        )

# Add predictions using pretrained model at different scales, on the same test images
gt_paths = np.unique(gt_paths).tolist()
for gt_path in gt_paths:
    for masks_path in dir_pretrained.glob(f"*{gt_path.stem.replace('-cells', '')}*tif"):
        factor = re.search(r"-ds(\d+)", masks_path.name).groups()[0]
        targets.append(
            ("cellposeSAM-pretrained", "pretrained", int(factor), gt_path, masks_path)
        )

data = pd.DataFrame(
    targets,
    columns=[
        "model_name",
        "downscaling_training",
        "downscaling_factor",
        "ground_truth_path",
        "prediction_path",
    ],
)

data = data.drop_duplicates(
    ["model_name", "downscaling_training", "downscaling_factor", "ground_truth_path"]
)
data = data.drop_duplicates(
    ["model_name", "downscaling_training", "downscaling_factor", "ground_truth_path"]
)
data["iou_threshold"] = [np.arange(start=0.05, stop=1, step=0.05)] * len(data)
data["average_precision"] = data.apply(calculate_ap, axis=1)

In [ ]:
data_exploded = data.explode(["iou_threshold", "average_precision"], ignore_index=True)
data_long = data_exploded.groupby(
    ["model_name", "downscaling_training", "downscaling_factor", "iou_threshold"],
    as_index=False,
    dropna=False,
)["average_precision"].aggregate("mean")
data_agg = data_long.groupby(
    ["model_name", "downscaling_training", "downscaling_factor"],
    as_index=False,
    dropna=False,
)["average_precision"].aggregate("mean")
data_agg = data_agg.rename(columns={"average_precision": "mean_average_precision"})

In [ ]:
# Performance at training resolutions
data_plot = data_long.query("downscaling_training==downscaling_factor")
g = sns.relplot(
    data_plot,
    x="iou_threshold",
    y="average_precision",
    hue="model_name",
    style="downscaling_training",
    kind="line",
)
g.ax.set(xlim=(0.05, 0.95), ylim=(0, 1))

In [ ]:
# Performance at downscaling 10
data_plot = data_long.query("downscaling_factor==[10]")
g = sns.relplot(
    data_plot,
    x="iou_threshold",
    y="average_precision",
    hue="model_name",
    style="downscaling_training",
    kind="line",
)
g.ax.set(xlim=(0.05, 0.95), ylim=(0, 1))

In [ ]:
data_plot = data_long.query(
    "(downscaling_training==downscaling_factor | downscaling_training=='pretrained')"
    + "& downscaling_factor==[5]"
)
g = sns.catplot(
    data_plot,
    x="downscaling_factor",
    y="average_precision",
    kind="bar",
    errorbar=None,
    hue="model_name",
)